# Student augmentation preview

Runs `dataloader.StudentAugment` on a 128-tile batch (one tile per random shard, so stain-mix donors come from other slides, as in a unique-slide batch) and shows 5 original/augmented pairs.

In [ ]:
import io
import random
import sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))

import matplotlib.pyplot as plt
import pyarrow.parquet as pq
import torch
from PIL import Image
from torchvision.transforms import v2

from dataloader import StudentAugment

DATA = {"dataset_dir": "/data/nanopath_parquet", "mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225]}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# data.student_augment block under test
AUG = {
    "hed_jitter": 0.02,
    "blur_prob": 0.5,
    "blur_sigma": [0.1, 1.0],
    "stain_mix": 1.0,
    "hue": 0.02,
    "rescale": 0.25,
    "jpeg": 1.0,
    "noise": 1.0,
}

In [ ]:
# Augment a full batch (op counts and stain donors depend on the batch), then plot n_show random original/augmented pairs.
def show_student_augment(aug_cfg, n_show=5, batch=128, seed=42):
    rng = random.Random(seed)
    random.seed(seed)  # StudentAugment draws its rescale factor from Python's global RNG
    torch.manual_seed(seed)
    # One tile per random shard, so the batch spans many slides and sites like a unique-slide training batch.
    shards = sorted(Path(DATA["dataset_dir"]).glob("shard-*.parquet"))
    to_tensor = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True), v2.Resize((224, 224), antialias=True)])
    tiles = []
    for f in rng.sample(shards, batch):
        pf = pq.ParquetFile(f)
        jpegs = pf.read_row_group(rng.randrange(pf.num_row_groups), columns=["jpeg"])["jpeg"].to_pylist()
        tiles.append(to_tensor(Image.open(io.BytesIO(rng.choice(jpegs))).convert("RGB")))
    x = torch.stack(tiles).to(device)
    mean, std = (torch.tensor(DATA[k], device=device).view(1, 3, 1, 1) for k in ("mean", "std"))
    # StudentAugment works on normalized views; b = batch makes the whole batch one donor slot.
    aug = (StudentAugment(DATA, **aug_cfg).to(device)((x - mean) / std, batch) * std + mean).clamp(0, 1)
    fig, axes = plt.subplots(2, n_show, figsize=(3 * n_show, 6.4))
    for col, i in enumerate(rng.sample(range(batch), n_show)):
        for row, (img, label) in enumerate(((x[i], "original"), (aug[i], "augmented"))):
            axes[row, col].imshow(img.permute(1, 2, 0).cpu().numpy())
            axes[row, col].set_title(f"tile {i} {label}", fontsize=9)
            axes[row, col].axis("off")
    fig.tight_layout()
    plt.show()


show_student_augment(AUG)